# 04 Results: local test set, external transportability, subgroup fairness, timing of death, temporal evaluation

Main text: Tables 1 and 2, Figures 2 to 4. Supplement: Tables S6 to S16, S19, S20 and Figures S1, S2, S4 to S6.
The per-municipality external results and the metric batteries are cached in `cache/`; delete the files to recompute.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_auc_score
from IPython.display import display
import shap
import utils

## Test cohorts, fitted models, frozen thresholds, predicted probabilities

In [ ]:
test_local = utils.load_object(utils.DATA + "/test_local_2016_moh")
test_ceinfo = utils.load_object(utils.DATA + "/test_ceinfo_2016")
temporal = utils.load_object(utils.DATA + "/test_temporal_sp_2017_2020")
test_cohorts = {"São Paulo 2016 local (MoH)": test_local,
                "São Paulo 2016 (CEInfo)": test_ceinfo}
for year in [2017, 2018, 2019, 2020]:
    test_cohorts["São Paulo " + str(year)] = temporal[temporal.ANONASC == year].reset_index(drop=True)
{name: cohort.shape for name, cohort in test_cohorts.items()}

In [ ]:
probabilities = {}
for model_name, label in utils.MODELS_ORDER:
    model = utils.load_object(utils.MODELS + "/model_" + model_name)
    probabilities[label] = {name: model.predict_proba(cohort[utils.FEATURES])[:, 1]
                            for name, cohort in test_cohorts.items()}
thresholds = {label: utils.load_object(utils.MODELS + "/thresholds_" + model_name)
              for model_name, label in utils.MODELS_ORDER}
model_labels = sorted(label for _, label in utils.MODELS_ORDER)   # alphabetical column order
local = "São Paulo 2016 local (MoH)"
y_local = test_local["OBITO"].values
prevalence = y_local.mean()
model_labels, round(float(prevalence), 5)

In [ ]:
# metric battery (point estimate and bootstrap 95% CI) for every model on every cohort
battery_path = utils.CACHE + "/battery.pkl"
if os.path.exists(battery_path):
    battery = utils.load_object(battery_path)
    print("reused cached battery")
else:
    battery = {}
    for name, cohort in test_cohorts.items():
        y = cohort["OBITO"].values
        battery[name] = {}
        for label in model_labels:
            battery[name][label] = utils.evaluate_with_confidence_intervals(
                y, probabilities[label][name], thresholds[label], n_bootstrap=1000)
        print("battery done:", name)
    utils.save_object(battery, battery_path)

## Table 1: discrimination, calibration and operating points on the local test set

In [ ]:
# Panel A: discrimination and native calibration
rows = []
for model_name, label in utils.MODELS_ORDER:
    lookup = {r["metric"]: r for _, r in battery[local][label].iterrows()}
    calibration = utils.calibration_in_the_large_and_slope(y_local, probabilities[label][local])
    rows.append({"Model": label,
                 "AUROC": utils.estimate_with_ci(lookup["auroc"]["value"], lookup["auroc"]["ci_low"], lookup["auroc"]["ci_high"]),
                 "AUPRC": utils.estimate_with_ci(lookup["auprc"]["value"], lookup["auprc"]["ci_low"], lookup["auprc"]["ci_high"]),
                 "Calibration intercept": round(calibration["calibration_in_the_large"], 3),
                 "Calibration slope": round(calibration["calibration_slope"], 3),
                 "Scaled Brier": round(lookup["scaled_brier"]["value"], 3)})
table1_panel_a = pd.DataFrame(rows)
table1_panel_a

In [ ]:
# Panel B: performance at the three operating points
operating_point_main = {"threshold 0.5": "Default (0.5)", "max sens + spec": "Max sens+spec",
                        "fixed sensitivity 0.80": "Fixed sens ≥0.80"}
rows = []
for model_name, label in utils.MODELS_ORDER:
    lookup = {r["metric"]: r for _, r in battery[local][label].iterrows()}
    for operating_point in ["threshold 0.5", "max sens + spec", "fixed sensitivity 0.80"]:
        row = {"Model": label, "Operating point": operating_point_main[operating_point]}
        for column, metric in [("Sensitivity", "sensitivity"), ("Specificity", "specificity"), ("PPV", "ppv"),
                               ("Proportion flagged", "alert_rate"), ("Balanced accuracy", "balanced_accuracy")]:
            r = lookup[metric + " @ " + operating_point]
            row[column] = utils.estimate_with_ci(r["value"], r["ci_low"], r["ci_high"])
        rows.append(row)
table1_panel_b = pd.DataFrame(rows)
table1_panel_b

## Figure 2: ROC and precision-recall curves, local test set

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 5))
roc_frames = []
for model_name, label in utils.MODELS_ORDER:
    color = utils.MODEL_COLORS[label]
    p = probabilities[label][local]
    lookup = {r["metric"]: r for _, r in battery[local][label].iterrows()}
    roc = utils.roc_points(y_local, p); roc["model"] = label
    pr = utils.precision_recall_points(y_local, p)
    a, b = lookup["auroc"], lookup["auprc"]
    ax_roc.plot(roc.false_positive_rate, roc.true_positive_rate, color=color, lw=1.8,
                label=label + "  (AUROC " + utils.estimate_with_ci(a["value"], a["ci_low"], a["ci_high"]) + ")")
    ax_pr.plot(pr.recall, pr.precision, color=color, lw=1.8,
               label=label + "  (AUPRC " + utils.estimate_with_ci(b["value"], b["ci_low"], b["ci_high"]) + ")")
    roc_frames.append(roc)
ax_roc.plot([0, 1], [0, 1], "--", color="0.6", lw=1, label="Chance (diagonal)")
ax_roc.set_xlabel("False positive rate (1 - specificity)"); ax_roc.set_ylabel("True positive rate (sensitivity)")
ax_roc.set_title("A. Receiver operating characteristic"); ax_roc.legend(loc="upper left", bbox_to_anchor=(0.0, -0.13), frameon=False)
ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.01)
ax_pr.axhline(prevalence, ls=":", color="0.5", lw=1, label="Prevalence baseline (" + str(round(prevalence, 3)) + ")")
ax_pr.set_xlabel("Recall (sensitivity)"); ax_pr.set_ylabel("Precision (PPV)")
ax_pr.set_title("B. Precision-recall"); ax_pr.legend(loc="upper left", bbox_to_anchor=(0.0, -0.13), frameon=False)
ax_pr.set_xlim(0, 1); ax_pr.set_ylim(0, 1.01)
utils.save_figure(fig, utils.FIGURES, "Figure_2_ROC_PR", data=pd.concat(roc_frames, ignore_index=True))
fig

## Figure 3: calibration and decision curve, local test set

In [ ]:
fig, (ax_cal, ax_dca) = plt.subplots(1, 2, figsize=(11, 5))
curve_max = 0
for model_name, label in utils.MODELS_ORDER:
    curve = utils.calibration_curve_points(y_local, probabilities[label][local])
    ax_cal.plot(curve.mean_predicted_probability, curve.observed_fraction_positive,
                "o-", color=utils.MODEL_COLORS[label], lw=1.6, ms=4, label=label)
    curve_max = max(curve_max, curve.mean_predicted_probability.max(), curve.observed_fraction_positive.max())
cal_lim = min(1.0, curve_max * 1.15)
ax_cal.plot([0, cal_lim], [0, cal_lim], "--", color="0.6", lw=1, label="Perfect calibration")
ax_cal.set_xlim(0, cal_lim); ax_cal.set_ylim(0, cal_lim)
ax_cal.set_xlabel("Mean predicted probability"); ax_cal.set_ylabel("Observed fraction positive")
ax_cal.set_title("A. Calibration (reliability)"); ax_cal.legend(loc="upper left", frameon=False)

net_benefit_frames = []
nb_max = 0
for model_name, label in utils.MODELS_ORDER:
    curve = utils.net_benefit_curve(y_local, probabilities[label][local])
    curve.insert(0, "model", label)
    net_benefit_frames.append(curve)
    ax_dca.plot(curve.threshold_probability, curve.net_benefit_model, color=utils.MODEL_COLORS[label], lw=1.8, label=label)
    nb_max = max(nb_max, curve.net_benefit_model.max())
ax_dca.plot(curve.threshold_probability, curve.net_benefit_treat_all, ":", color="0.55", lw=1.1, label="Treat all")
ax_dca.plot(curve.threshold_probability, curve.net_benefit_treat_none, "-", color="0.35", lw=1.1, label="Treat none")
ax_dca.axvline(prevalence, ls="--", color="0.45", lw=1.0, label="Prevalence (" + str(round(prevalence, 3)) + ")")
ax_dca.set_xlim(0, 0.03); ax_dca.set_ylim(-0.0006, nb_max * 1.12); ax_dca.margins(x=0)
ax_dca.axhline(0, color="0.85", lw=0.6, zorder=0)
ax_dca.set_xlabel("Threshold probability"); ax_dca.set_ylabel("Net benefit")
ax_dca.set_title("B. Decision curve (net benefit)"); ax_dca.legend(loc="center right", frameon=False)
fig.tight_layout()
utils.save_figure(fig, utils.FIGURES, "Figure_3_calibration_decision", data=pd.concat(net_benefit_frames, ignore_index=True))
fig

## Table S6: full metric battery on the local test set (MoH and CEInfo linkages)

In [ ]:
sheets_s6 = {}
for name in ["São Paulo 2016 local (MoH)", "São Paulo 2016 (CEInfo)"]:
    sheets_s6[name] = utils.metric_battery_wide({label: battery[name][label] for label in model_labels})
utils.save_sheets(sheets_s6, utils.SUPPLEMENT_TABLES, "Table_S6")
for sheet, table in sheets_s6.items():
    print(sheet)
    display(table)

## Table S7: AUROC comparison with the logistic regression baseline (paired bootstrap and DeLong test)

In [ ]:
n_other = 3   # three models compared against the baseline, for the Bonferroni correction
sheets_s7 = {}
for name, cohort in test_cohorts.items():
    y = cohort["OBITO"].values
    reference = probabilities["Logistic regression"][name]
    rows = []
    for label in ["XGBoost", "LightGBM", "CatBoost"]:
        p = probabilities[label][name]
        difference = utils.paired_bootstrap_auroc_difference(y, p, reference)
        delong = utils.delong_test(y, p, reference)
        rows.append({"Model": label, "Reference": "Logistic regression",
                     "AUROC difference [95% CI]": utils.estimate_with_ci(
                         difference["auroc_difference"], difference["ci_low"], difference["ci_high"]),
                     "DeLong p (Bonferroni)": round(min(1.0, delong["p_value"] * n_other), 2)})
    sheets_s7[name] = pd.DataFrame(rows)
utils.save_sheets(sheets_s7, utils.SUPPLEMENT_TABLES, "Table_S7")
for sheet, table in sheets_s7.items():
    print(sheet)
    display(table)

## Table S8: calibration (native, sigmoid, isotonic) per cohort

In [ ]:
sheets_s8 = {}
for name, cohort in test_cohorts.items():
    y = cohort["OBITO"].values
    rows = []
    for model_name, label in utils.MODELS_ORDER:
        p = probabilities[label][name]
        calibrators = utils.load_object(utils.MODELS + "/calibrators_" + model_name)
        for version, values in [("native", p),
                                ("sigmoid", calibrators["sigmoid"].predict(p)),
                                ("isotonic", calibrators["isotonic"].predict(p))]:
            summary = utils.calibration_in_the_large_and_slope(y, values)
            rows.append({"Model": label, "Version": version,
                         "Calibration-in-the-large": round(summary["calibration_in_the_large"], 3),
                         "Calibration slope": round(summary["calibration_slope"], 3),
                         "Scaled Brier": round(utils.scaled_brier_score(y, values), 3)})
    sheets_s8[name] = pd.DataFrame(rows)
utils.save_sheets(sheets_s8, utils.SUPPLEMENT_TABLES, "Table_S8")
for sheet, table in sheets_s8.items():
    print(sheet)
    display(table)

## Table S9, Figure 4, Figures S1 and S2: SHAP predictor importance, local test set

In [ ]:
shap_sample = test_local[utils.FEATURES].sample(n=min(5000, len(test_local)), random_state=utils.SEED)
importance = {}
for model_name, label in utils.MODELS_ORDER:
    model = utils.load_object(utils.MODELS + "/model_" + model_name)
    importance[label] = utils.shap_mean_abs_importance(model, shap_sample)   # indexed by predictor key
shap_importance = pd.DataFrame(importance)
shap_importance

In [ ]:
# Table S9: mean absolute SHAP per predictor and model
rows = []
for model_name, label in utils.MODELS_ORDER:
    ordered = shap_importance[label].sort_values(ascending=False)
    for variable, value in ordered.items():
        rows.append({"variable": utils.VARIABLE_LABELS[variable], "mean absolute shap": round(value, 4), "model": label})
table_s9 = pd.DataFrame(rows)
utils.save_table(table_s9, utils.SUPPLEMENT_TABLES, "Table_S9")
table_s9.head(8)

In [ ]:
# Figure 4: relative predictor importance across the four models (share of each model's total mean |SHAP|)
shares = shap_importance / shap_importance.sum()
order = shares.mean(axis=1).sort_values().index
shares = shares.loc[order]
positions = np.arange(len(order)); height = 0.2
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for i, (model_name, label) in enumerate(utils.MODELS_ORDER):
    ax.barh(positions + (i - 1.5) * height, shares[label].values, height=height,
            color=utils.MODEL_COLORS[label], label=label)
ax.set_yticks(positions); ax.set_yticklabels([utils.VARIABLE_LABELS[v] for v in order])
ax.set_xlabel("Relative predictor importance (share of each model's total mean |SHAP|)")
ax.legend(frameon=False, loc="lower right")
utils.save_figure(fig, utils.FIGURES, "Figure_4_SHAP", data=shares.reset_index())
fig

In [ ]:
# Figure S1: mean absolute SHAP per predictor, one panel per model
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (model_name, label) in zip(axes.ravel(), utils.MODELS_ORDER):
    ordered = shap_importance[label].sort_values(ascending=True)
    ax.barh([utils.VARIABLE_LABELS[v] for v in ordered.index], ordered.values)
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title("Feature importance - " + label)
fig.tight_layout()
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S1_SHAP_bar_by_model")
fig

In [ ]:
# Figure S2: SHAP summary (beeswarm) on the preprocessed columns, one panel per model
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for ax, (model_name, label) in zip(axes.ravel(), utils.MODELS_ORDER):
    model = utils.load_object(utils.MODELS + "/model_" + model_name)
    values, transformed, names = utils.shap_values_for_beeswarm(model, shap_sample)
    plt.sca(ax)
    shap.summary_plot(values, transformed, feature_names=[n.split("__", 1)[-1] for n in names],
                      plot_type="dot", cmap="coolwarm", max_display=12, show=False, plot_size=None)
    ax.set_title("SHAP summary - " + label)
fig.subplots_adjust(left=0.13, right=0.93, top=0.95, bottom=0.07, wspace=0.62, hspace=0.30)
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S2_SHAP_beeswarm_by_model")
fig

## Tables S10 to S13 and Table 2: external transportability across Brazilian municipalities

In [ ]:
external = utils.load_object(utils.DATA + "/test_external_brazil_2016")
external_y = external["OBITO"].astype(int).values
city_index = external.groupby("CODMUNNASC").indices
external.shape, int(external_y.sum()), len(city_index)

In [ ]:
# municipality population in 2016 (IBGE estimates; six-digit code = CODMUNNASC)
ibge = pd.read_csv(utils.RAW + "/ibge_cnv_popsvs2024.csv", sep=";", encoding="latin-1",
                   skiprows=3, names=["municipality", "2016", "2017", "2018", "2019", "2020"])
ibge = ibge[ibge["municipality"].str.match(r"^\s*\d{6}", na=False)]
ibge["code"] = ibge["municipality"].str.extract(r"(\d{6})").astype(int)
ibge["population_2016"] = pd.to_numeric(ibge["2016"], errors="coerce")
population_by_code = dict(zip(ibge["code"], ibge["population_2016"]))
len(population_by_code)

In [ ]:
# per-municipality AUROC for each model (cached)
per_city = {}
for model_name, label in utils.MODELS_ORDER:
    path = utils.CACHE + "/external_per_city_" + model_name + ".pkl"
    if os.path.exists(path):
        per_city[label] = utils.load_object(path)
    else:
        model = utils.load_object(utils.MODELS + "/model_" + model_name)
        probability = model.predict_proba(external[utils.FEATURES])[:, 1]
        rows = []
        for city, index in city_index.items():
            rows.append({"municipality": int(city),
                         "population_group": utils.population_size_group(population_by_code.get(int(city))),
                         "n": len(index), "deaths": int(external_y[index].sum()),
                         "auroc": utils.safe_auroc(external_y[index], probability[index])})
        per_city[label] = {"per_city": pd.DataFrame(rows),
                           "pooled_auroc": utils.safe_auroc(external_y, probability)}
        utils.save_object(per_city[label], path)
    frame = per_city[label]["per_city"]
    per_city[label]["per_city"] = frame[frame.municipality.isin(city_index.keys())].reset_index(drop=True)
{label: len(per_city[label]["per_city"]) for label in per_city}

In [ ]:
# Table S10: median per-municipality AUROC by population size, one sheet per model
sheets_s10 = {}
for model_name, label in utils.MODELS_ORDER:
    frame = per_city[label]["per_city"]
    evaluable = frame[frame.deaths >= 2].dropna(subset=["auroc"])
    rows = []
    for group in utils.POPULATION_SIZE_ORDER:
        cities = evaluable[evaluable.population_group == group]
        rows.append({"Population size (inhabitants)": group,
                     "Municipalities (total)": int((frame.population_group == group).sum()),
                     "Municipalities (evaluable)": int(len(cities)),
                     "Neonatal deaths": int(frame[frame.population_group == group].deaths.sum()),
                     "Median per-site AUROC": round(cities.auroc.median(), 3) if len(cities) else np.nan})
    sheets_s10[label] = pd.DataFrame(rows)
utils.save_sheets(sheets_s10, utils.SUPPLEMENT_TABLES, "Table_S10")
for sheet, table in sheets_s10.items():
    print(sheet)
    display(table)

In [ ]:
# Table S11: record-pooled AUROC and per-municipality median [IQR]
rows = []
for model_name, label in utils.MODELS_ORDER:
    frame = per_city[label]["per_city"]
    evaluable = frame[frame.deaths >= 2].dropna(subset=["auroc"])
    median_iqr = (str(round(evaluable.auroc.median(), 3)) + " ["
                  + str(round(evaluable.auroc.quantile(0.25), 3)) + "-"
                  + str(round(evaluable.auroc.quantile(0.75), 3)) + "]")
    rows.append({"Model": label,
                 "Record-pooled AUROC": round(per_city[label]["pooled_auroc"], 3),
                 "Municipalities (evaluable)": int(len(evaluable)),
                 "Per-site AUROC, median [IQR]": median_iqr})
table_s11 = pd.DataFrame(rows)
utils.save_table(table_s11, utils.SUPPLEMENT_TABLES, "Table_S11")
table_s11

In [ ]:
# Table S12: pooled external calibration, native, sigmoid and isotonic probabilities (cached)
pooled_path = utils.CACHE + "/external_pooled_calibration.pkl"
if os.path.exists(pooled_path):
    table_s12 = utils.load_object(pooled_path)
else:
    rows = []
    for model_name, label in utils.MODELS_ORDER:
        model = utils.load_object(utils.MODELS + "/model_" + model_name)
        probability = model.predict_proba(external[utils.FEATURES])[:, 1]
        calibrators = utils.load_object(utils.MODELS + "/calibrators_" + model_name)
        for version, values in [("native", probability),
                                ("sigmoid", calibrators["sigmoid"].predict(probability)),
                                ("isotonic", calibrators["isotonic"].predict(probability))]:
            summary = utils.calibration_in_the_large_and_slope(external_y, values)
            rows.append({"Model": label, "Probability version": version,
                         "Calibration-in-the-large": round(summary["calibration_in_the_large"], 3),
                         "Calibration slope": round(summary["calibration_slope"], 3),
                         "Scaled Brier": round(utils.scaled_brier_score(external_y, values), 3)})
    table_s12 = pd.DataFrame(rows)
    utils.save_object(table_s12, pooled_path)
utils.save_table(table_s12, utils.SUPPLEMENT_TABLES, "Table_S12")
table_s12

In [ ]:
# Table S13: failing tail (bottom decile of the CatBoost per-municipality AUROC among evaluable municipalities)
# versus the rest; per-municipality rows carry the case-mix divergence from the training cohort and the
# per-municipality calibration-in-the-large (cached)
tail_path = utils.CACHE + "/external_failing_tail_per_municipality.pkl"
if os.path.exists(tail_path):
    municipalities = utils.load_object(tail_path)
else:
    train = utils.load_object(utils.DATA + "/train_2012_2014")
    train_mean = train[utils.CONTINUOUS].mean()
    train_std = train[utils.CONTINUOUS].replace(0, np.nan).std().replace(0, np.nan)
    main_model = utils.load_object(utils.MODELS + "/model_catboost")
    external_probability = main_model.predict_proba(external[utils.FEATURES])[:, 1]
    auroc_by_city = dict(zip(per_city["CatBoost"]["per_city"].municipality, per_city["CatBoost"]["per_city"].auroc))
    rows = []
    for city, index in city_index.items():
        y = external_y[index]
        if y.sum() < 1:
            continue
        block = external.iloc[index]
        rows.append({"municipality": int(city), "n": len(index), "deaths": int(y.sum()),
                     "population": population_by_code.get(int(city)),
                     "population_group": utils.population_size_group(population_by_code.get(int(city))),
                     "case_mix_divergence": utils.case_mix_divergence(block[utils.CONTINUOUS], train_mean, train_std),
                     "calibration_in_the_large": utils.calibration_in_the_large_and_slope(
                         y, external_probability[index])["calibration_in_the_large"],
                     "auroc": auroc_by_city.get(int(city))})
    municipalities = pd.DataFrame(rows)
    utils.save_object(municipalities, tail_path)

municipalities = municipalities.rename(columns={"auroc": "auroc_catboost"})
municipalities["evaluable"] = (municipalities.deaths >= 2).astype(int)
municipalities.loc[municipalities.evaluable == 0, "auroc_catboost"] = np.nan
evaluable = municipalities[municipalities.evaluable == 1]
tail_threshold = evaluable.auroc_catboost.quantile(0.10)
municipalities["failing_tail"] = np.where(municipalities.evaluable == 1,
                                          (municipalities.auroc_catboost < tail_threshold).astype(float), np.nan)
tail = evaluable[evaluable.auroc_catboost < tail_threshold]
rest = evaluable[evaluable.auroc_catboost >= tail_threshold]
print("bottom-decile threshold:", round(tail_threshold, 4), "| failing tail:", len(tail), "| rest:", len(rest))

summary = []
for name, group in [("Failing tail", tail), ("Rest", rest)]:
    summary.append({"Group": name, "Municipalities, n": len(group),
                    "Per-site AUROC, median (range)": f"{group.auroc_catboost.median():.3f} ({group.auroc_catboost.min():.3f}-{group.auroc_catboost.max():.3f})",
                    "Population, median": int(group.population.median()),
                    "Live births, median": int(group.n.median()),
                    "Neonatal deaths, median": int(group.deaths.median()),
                    "Case-mix divergence, median": round(group.case_mix_divergence.median(), 3),
                    "Calibration-in-the-large, median": round(group.calibration_in_the_large.median(), 3),
                    "Population <= 20,000, n (%)": f"{int((group.population <= 20000).sum())} ({100 * (group.population <= 20000).mean():.1f})",
                    "Population > 100,000, n (%)": f"{int((group.population > 100000).sum())} ({100 * (group.population > 100000).mean():.1f})"})
table_s13 = pd.DataFrame(summary).set_index("Group").T.reset_index().rename(columns={"index": "Characteristic"})
utils.save_sheets({"Failing tail vs rest": table_s13, "Municipalities": municipalities}, utils.SUPPLEMENT_TABLES, "Table_S13")
table_s13

In [ ]:
# per-municipality results of all four models (supporting data file)
frames = []
for model_name, label in utils.MODELS_ORDER:
    one = per_city[label]["per_city"].copy()
    one.insert(0, "model", label)
    frames.append(one[one.deaths >= 1])
utils.save_table(pd.concat(frames, ignore_index=True), utils.SUPPLEMENT_TABLES, "per_municipality_auroc_all_models")

In [ ]:
# Table 2: median per-municipality AUROC by population size, all four models, with an overall row
size_label = {"up to 5,000": "≤ 5,000", "5,001 to 10,000": "5,001-10,000",
              "10,001 to 20,000": "10,001-20,000", "20,001 to 100,000": "20,001-100,000",
              "100,001 to 500,000": "100,001-500,000", "more than 500,000": "> 500,000"}
model_column = {"Logistic regression": "Logistic regression baseline"}
reference_frame = per_city["CatBoost"]["per_city"]

rows = []
for group in utils.POPULATION_SIZE_ORDER:
    total = int((reference_frame.population_group == group).sum())
    deaths = int(reference_frame[reference_frame.population_group == group].deaths.sum())
    evaluable_n = int(((reference_frame.population_group == group) & (reference_frame.deaths >= 2)).sum())
    row = {"Population size (inhabitants)": size_label[group], "Municipalities (total)": total,
           "Municipalities evaluable": evaluable_n, "Neonatal deaths": deaths}
    for model_name, label in utils.MODELS_ORDER:
        cities = per_city[label]["per_city"]
        cities = cities[(cities.population_group == group) & (cities.deaths >= 2)].dropna(subset=["auroc"])
        row[model_column.get(label, label)] = round(cities.auroc.median(), 3) if len(cities) else "not estimable"
    rows.append(row)

overall = {"Population size (inhabitants)": "All municipalities (median [IQR])",
           "Municipalities (total)": int(len(reference_frame)),
           "Municipalities evaluable": int((reference_frame.deaths >= 2).sum()),
           "Neonatal deaths": int(reference_frame.deaths.sum())}
for model_name, label in utils.MODELS_ORDER:
    cities = per_city[label]["per_city"]
    evaluable_cities = cities[cities.deaths >= 2].dropna(subset=["auroc"])
    overall[model_column.get(label, label)] = (str(round(evaluable_cities.auroc.median(), 3)) + " ["
        + str(round(evaluable_cities.auroc.quantile(0.25), 3)) + "-" + str(round(evaluable_cities.auroc.quantile(0.75), 3)) + "]")
table2 = pd.DataFrame(rows + [overall])
table2

In [ ]:
utils.save_sheets({"Table 1 Panel A": table1_panel_a, "Table 1 Panel B": table1_panel_b, "Table 2": table2},
                  utils.TABLES, "Tables_1-2_main")

## Tables S14 to S16: subgroup fairness on the local test set

In [ ]:
masks = utils.subgroup_masks(test_local)
dimension_sheet = {"race": "Race", "education": "Education", "maternal_age": "Maternal Age"}

# Table S14: full metric battery per subgroup and model
sheets_s14 = {}
for dimension, sheet_name in dimension_sheet.items():
    columns = {}
    for model_name, label in utils.MODELS_ORDER:
        p = probabilities[label][local]
        for group_name, mask in masks[dimension].items():
            index = np.where(mask.values)[0]
            columns[label + " — " + group_name] = utils.evaluate_with_confidence_intervals(
                y_local[index], p[index], thresholds[label])
    sheets_s14[sheet_name] = utils.metric_battery_wide(columns)
utils.save_sheets(sheets_s14, utils.SUPPLEMENT_TABLES, "Table_S14")
for sheet, table in sheets_s14.items():
    print(sheet)
    display(table)

In [ ]:
# Table S15: equal-opportunity (sensitivity) and equalized-odds (false-positive-rate) gaps with bootstrap CIs
sheets_s15 = {}
for dimension, sheet_name in dimension_sheet.items():
    (group_a, mask_a), (group_b, mask_b) = list(masks[dimension].items())
    rows = []
    for model_name, label in utils.MODELS_ORDER:
        p = probabilities[label][local]
        for operating_point, threshold in thresholds[label].items():
            gaps = utils.fairness_gaps_with_ci(y_local, p, mask_a.values, mask_b.values, threshold)
            for gap_type in ["sensitivity", "fpr"]:
                rows.append({"Model": label, "Comparison": group_a + " - " + group_b,
                             "Operating point": operating_point, "Gap type": gap_type,
                             "Gap [95% CI]": utils.estimate_with_ci(
                                 gaps[gap_type]["gap"], gaps[gap_type]["ci_low"], gaps[gap_type]["ci_high"])})
    sheets_s15[sheet_name] = pd.DataFrame(rows)
utils.save_sheets(sheets_s15, utils.SUPPLEMENT_TABLES, "Table_S15")
for sheet, table in sheets_s15.items():
    print(sheet)
    display(table)

In [ ]:
# Table S16: three-way intersection (race x education x maternal age) at the max sens + spec point
sheets_s16 = {}
for model_name, label in utils.MODELS_ORDER:
    p = probabilities[label][local]
    threshold = thresholds[label]["max sens + spec"]
    rows = []
    for race_name, race_mask in masks["race"].items():
        for education_name, education_mask in masks["education"].items():
            for age_name, age_mask in masks["maternal_age"].items():
                index = np.where((race_mask & education_mask & age_mask).values)[0]
                if len(index) == 0:
                    continue
                y = y_local[index]
                counts = utils.confusion_matrix_at(y, p[index], threshold)
                sensitivity = utils.classification_metrics_from_counts(counts)["sensitivity"]
                rows.append({"Subgroup cell": race_name + " & " + education_name + " & " + age_name,
                             "N": len(index), "Deaths": int(y.sum()),
                             "AUROC": round(utils.safe_auroc(y, p[index]), 3),
                             "Sensitivity (max sens + spec)": round(sensitivity, 3)})
    sheets_s16[label] = pd.DataFrame(rows)
utils.save_sheets(sheets_s16, utils.SUPPLEMENT_TABLES, "Table_S16")
for sheet, table in sheets_s16.items():
    print(sheet)
    display(table)

## Table S19, Figures S4 and S5: detection by timing of death (one-versus-rest), local test set

In [ ]:
timing = test_local["death_timing"].values
sheets_s19 = {}
for model_name, label in utils.MODELS_ORDER:
    p = probabilities[label][local]
    columns = {}
    for window in utils.DEATH_TIMING_ORDER:
        rows = utils.death_timing_window_rows(y_local, timing, window)
        columns[window] = utils.evaluate_with_confidence_intervals(y_local[rows], p[rows], thresholds[label])
    sheets_s19[label] = utils.metric_battery_wide(columns)
utils.save_sheets(sheets_s19, utils.SUPPLEMENT_TABLES, "Table_S19")
for sheet, table in sheets_s19.items():
    print(sheet)
    display(table)

In [ ]:
# Figure S4: ROC by death-timing window, one panel per model
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (model_name, label) in zip(axes.ravel(), utils.MODELS_ORDER):
    p = probabilities[label][local]
    for window in utils.DEATH_TIMING_ORDER:
        rows = utils.death_timing_window_rows(y_local, timing, window)
        roc = utils.roc_points(y_local[rows], p[rows])
        ax.plot(roc.false_positive_rate, roc.true_positive_rate, label=window)
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("ROC by death-timing window - " + label)
    ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S4_ROC_by_death_timing")
fig

In [ ]:
# Figure S5: precision-recall by death-timing window, one panel per model
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (model_name, label) in zip(axes.ravel(), utils.MODELS_ORDER):
    p = probabilities[label][local]
    for window in utils.DEATH_TIMING_ORDER:
        rows = utils.death_timing_window_rows(y_local, timing, window)
        pr = utils.precision_recall_points(y_local[rows], p[rows])
        ax.plot(pr.recall, pr.precision, label=window)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-recall by death-timing window - " + label)
    ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S5_PR_by_death_timing")
fig

## Table S20 and Figure S6: within-city temporal evaluation, São Paulo 2017-2020

In [ ]:
sheets_s20 = {}
for year in [2017, 2018, 2019, 2020]:
    name = "São Paulo " + str(year)
    sheets_s20[name] = utils.metric_battery_wide({label: battery[name][label] for label in model_labels})
utils.save_sheets(sheets_s20, utils.SUPPLEMENT_TABLES, "Table_S20")
for sheet, table in sheets_s20.items():
    print(sheet)
    display(table)

In [ ]:
years = [2017, 2018, 2019, 2020]
fig, ax = plt.subplots(figsize=(8, 5))
trend_rows = []
for model_name, label in utils.MODELS_ORDER:
    aurocs = [roc_auc_score(test_cohorts["São Paulo " + str(y)]["OBITO"].values,
                            probabilities[label]["São Paulo " + str(y)]) for y in years]
    ax.plot(years, aurocs, marker="o", color=utils.MODEL_COLORS[label], label=label)
    trend_rows.append(pd.DataFrame({"model": label, "year": years, "auroc": aurocs}))
ax.set_xlabel("Year"); ax.set_ylabel("AUROC"); ax.set_xticks(years); ax.legend()
utils.save_figure(fig, utils.SUPPLEMENT_FIGURES, "Figure_S6_temporal_AUROC",
                  data=pd.concat(trend_rows, ignore_index=True))
fig